ベース：黒地図 CARTO Dark Matter

国境線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin0.geojson

地域線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

人口分布：WorldPop 'Syrian Arab Republic 100m Population 2026'
https://hub.worldpop.org/geodata/summary?id=75632
syr_worldpop_2026.tif

カラーコード：QGIS

tifデータの軽量化：pop_array[::2, ::2]

QGISの設計値をMatplotlibのカラーマップに落とし込み、Brancaを用いてHTML上のカラーバーとして再設計。

In [11]:
import rasterio

import folium
from folium import plugins  

import geopandas as gpd

import matplotlib.colors as mcolors 

# カラーバーを作るためのbranca
import branca 

In [12]:
# ETL Extract
pop_data = 'tif_1_syria_worldpop_2026.tif'

with rasterio.open(pop_data) as src:
        
    # ETL Extract
    # tifデータから、シリアの緯度経度を取得する
    pop_array = src.read(1)

    # ETL Transform downsampling
    # 膨大すぎる光の粒（ピクセル情報）を1つ飛ばしで間引く(4分の1に軽減)
    pop_array = pop_array[::2, ::2]
    
    # 
    bounds = [[src.bounds.bottom, src.bounds.left], [src.bounds.top, src.bounds.right]]

In [13]:
# QGIS　オリジナルカラーコード 黒に映えるネオンカラー

colors = [
    (0.00, "#00176100"), 
    (0.20, "#297b8eff"), 
    (0.40, "#28ae80ff"), 
    (0.60, "#2eb37cff"), 
    (0.80, "#e5e419ff"), 
    (1.00, "#fde725ff")  
]

marisa_cmap = mcolors.LinearSegmentedColormap.from_list("Syria_Neon", colors)

In [14]:
# 黒地図ベース

tiles_dark_nolabels = 'https://{s}.basemaps.cartocdn.com/dark_nolabels/{z}/{x}/{y}{r}.png'
attr_dark = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

m = folium.Map(location=[34.8, 38.5], zoom_start=7, tiles=tiles_dark_nolabels, attr=attr_dark)

In [15]:
# 国境線　太さ1
folium.GeoJson('syr_admin0.geojson', name='Country',
               style_function=lambda x: {'color': 'white', 'weight': 1, 'fillOpacity': 0}).add_to(m)

# 地域線　太さ1
folium.GeoJson('syr_admin1.geojson', name='Governorate',
               style_function=lambda x: {'color': 'white', 'weight': 1, 'fillOpacity': 0}).add_to(m)


In [16]:
# ETL Transform
# 編集した4つの条件を統合させる
# 光の粒、緯度経度の位置、QGISで決めたカラーコード、透け感は0.8

img = folium.raster_layers.ImageOverlay(
    image=pop_array,
    bounds=bounds,
    colormap=marisa_cmap,
    opacity=0.8)

# ETL Load
img.add_to(m)

In [17]:
# 隣国の名入れ

neighbors = {
    "TÜRKIYE": [37.5, 37.5],      
    "IRAQ": [34.5, 42.0],         
    "JORDAN": [31.9, 36.5],       
    "LEBANON": [34.2, 35.0]       
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size: 14pt; 
                font-weight: bold; 
                color: gray; 
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
                ">{name}</div>'''
        )
    ).add_to(m)

In [18]:
# シリア14地域の名入れ

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size: 10pt; 
                color: white; 
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
                ">{name}</div>'''
        )
    ).add_to(m)

In [19]:
# カラーバー作業台

colormap = branca.colormap.LinearColormap(
    colors=[marisa_cmap(i/5) for i in range(6)], 
    index=[0, 20, 40, 60, 80, 100],               
    vmin=0, vmax=100,
    caption='Estimated Population per 100m cell'
)
m.add_child(colormap)

dark_legend_css = """
<style>
    .legend {
        background: rgba(0, 0, 0, 0.7) !important;
        color: white !important;
        padding: 12px !important;
        border-radius: 8px !important;
        border: 1px solid rgba(255, 255, 255, 0.4) !important;
    }

    .legend text {
        fill: white !important;
        font-weight: bold !important;
        font-size: 12px !important;
    }
    .legend caption {
        color: white !important;
        font-weight: bold !important;
    }
</style>
"""
m.get_root().header.add_child(folium.Element(dark_legend_css))

title_html = '''
             <div style="position: fixed; 
                         top: 20px; left: 50px; width: 360px; height: 150px; 
                         background-color: rgba(0,0,0,0.75); color: white;
                         z-index:9000; font-size:14px;
                         border:1px solid white; border-radius:8px; padding: 12px;
                         box-shadow: 0 0 15px rgba(0,0,0,0.5);">
             <b style="font-size: 16px;">Syrian Arab Republic</b><br>
             <span style="color: #ffd700;">Population Density (2026 Estimated)</span><br>
             <small style="display: block; margin-top: 5px; line-height: 1.2; color: #eee;">
                Estimated total number of people per grid-cell at a resolution of 3 arc seconds (approx. 100m).
             </small>
             <div style="margin-top: 10px; font-size: 11px; color: #bbb; border-top: 1px solid #444; padding-top: 5px;">
             Source: <a href="https://hub.worldpop.org/geodata/summary?id=75632" target="_blank" style="color: #3498db; text-decoration: none;">WorldPop (Open Access Data)</a>
             </div>
             </div>
             '''
m.get_root().html.add_child(folium.Element(title_html))


In [ ]:
# ETL Load

m.save('05_Syria_dmap_pop2026.html')